# 03 · Tokenization and embeddings: guided walkthrough

This follows **your** notebook (`01_tokenization_and_embeddings.ipynb`) section by section, on a bigger
corpus (`data/raw/input.txt`, Tiny Shakespeare), and ends with the pieces our project rebuilds as real modules.

**How to use it**
- Run top to bottom with Shift+Enter. Plain cells are examples. Read the output before moving on.
- 🔮 **Predict** cells: write your guess down first.
- ✍️ **YOU WRITE THIS** cells are the key code. Each one is followed by a ✅/❌ check cell. `⏳` means "not written yet".
- Reference solutions are in `notebooks/solutions/03_solutions.py`. Try for real before looking.

| Section | Your notebook | You write | Becomes (project) |
|---|---|---|---|
| 1 Word-level tokens | Steps 1–2, `SimpleTokenizerV2` | – | (motivation) |
| 2 Byte-pair encoding | tiktoken | **E1** BPE from scratch | `tokenizer/`, learning-plan item 2 |
| 3 Next-token targets | x / y shift | **E2** `make_xy` | the LM objective |
| 4 Sliding windows | `GPTDatasetV1` | **E3** `sliding_windows` | `SlidingWindowDataset` |
| 5 Memmap sampling | (new) | **E4** `sample_batch` | `data/dataset.py`, learning-plan item 1 |
| 6 Embeddings | `nn.Embedding`, positions | **E5**, **E6** | `model/gpt.py` input stage |

In [1]:
import re
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np
import tiktoken
import torch
from torch.utils.data import Dataset

from tinygpt.tokenizer import train_bpe

raw_text = Path("../data/raw/input.txt").read_text(encoding="utf-8")
print(f"{len(raw_text):,} characters")
print(raw_text[:250])
gpt2 = tiktoken.get_encoding("gpt2")


def check(label, fn):
    """Run one check; ✅ pass, ❌ wrong answer, ⏳ not implemented, 💥 crashed."""
    try:
        fn()
    except NotImplementedError:
        print(f"⏳ {label}: not implemented yet")
        return False
    except AssertionError as e:
        print(f"❌ {label}: {e}")
        return False
    except Exception as e:  # noqa: BLE001
        print(f"💥 {label}: {type(e).__name__}: {e}")
        return False
    print(f"✅ {label}")
    return True

1,115,394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



## 1 · Word-level tokens (your Steps 1–2)

Your notebook split text with a regex, built `vocab = sorted(set(tokens))`, and needed `<|unk|>` for anything new.
Same thing on Shakespeare:

In [2]:
pattern = r'([,.:;?_!"()\']|--|\s)'
tokens = [t.strip() for t in re.split(pattern, raw_text) if t.strip()]
print(len(tokens), "tokens;", len(set(tokens)), "distinct")
print(tokens[:25])

vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)) + ["<|endoftext|>", "<|unk|>"])}


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        toks = [t.strip() for t in re.split(pattern, text) if t.strip()]
        return [self.str_to_int.get(t, self.str_to_int["<|unk|>"]) for t in toks]

    def decode(self, ids):
        return " ".join(self.int_to_str[i] for i in ids)


v2 = SimpleTokenizerV2(vocab)
s = "Once upon a time, a hobbit sang of the Internet."
print(v2.decode(v2.encode(s)))

260467 tokens; 13851 distinct
['First', 'Citizen', ':', 'Before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'All', ':', 'Speak', ',', 'speak', '.', 'First', 'Citizen', ':', 'You', 'are', 'all']
Once upon a time , a <|unk|> <|unk|> of the <|unk|> .


**Problem:** `hobbit` and `Internet` became `<|unk|>` (information destroyed), the vocab is ~14k entries
(embedding rows), and punctuation spacing is lost. Byte-level BPE fixes all three.

## 2 · Byte-pair encoding (your tiktoken section)

First the tool you used, plus our own tokenizer trained on this corpus.

In [3]:
for s in ["Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.", "Akwirw ier"]:
    ids = gpt2.encode(s, allowed_special={"<|endoftext|>"})
    print(ids)
    print([gpt2.decode([i]) for i in ids])
    assert gpt2.decode(ids) == s

enc_text = gpt2.encode(raw_text)
ours = train_bpe(raw_text.split("\n\n"), vocab_size=1024)
ours_ids = ours.encode(raw_text)

print(f"\ncharacters:           {len(raw_text):>9,}")
print(f"word-level tokens:    {len(tokens):>9,}  (vocab {len(vocab):,})")
print(f"gpt2 BPE tokens:      {len(enc_text):>9,}  (vocab {gpt2.n_vocab:,})")
print(f"our BPE-1024 tokens:  {len(ours_ids):>9,}  (vocab {ours.vocab_size:,})")
print(f"chars per token: gpt2 {len(raw_text)/len(enc_text):.2f}, ours {len(raw_text)/len(ours_ids):.2f}")

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
['Hello', ',', ' do', ' you', ' like', ' tea', '?', ' ', '<|endoftext|>', ' In', ' the', ' sun', 'lit', ' terr', 'aces', ' of', ' some', 'unknown', 'Place', '.']
[33901, 86, 343, 86, 220, 959]
['Ak', 'w', 'ir', 'w', ' ', 'ier']



characters:           1,115,394
word-level tokens:      260,467  (vocab 13,853)
gpt2 BPE tokens:        338,025  (vocab 50,257)
our BPE-1024 tokens:    459,913  (vocab 1,024)
chars per token: gpt2 3.30, ours 2.43


🔮 **Predict:** how many tokens will our 1024-vocab BPE need compared with GPT-2's 50k vocab? About the same, 1.5x, or 3x?
(Answer is in the output above: a 50x smaller vocab costs only a modest increase in sequence length.)

### ✍️ E1 · Learn BPE yourself (pure Python)
This is the gap in your original notebook: it *used* BPE but never *learned* it. The algorithm:

1. `ids = list(text.encode("utf-8"))` (numbers 0–255)
2. count adjacent pairs, take the most frequent pair
3. replace every occurrence with a new id `256 + k`, remember the pair
4. repeat `n_merges` times. Encoding replays the merges **in learned order**.

Write the five functions. Hint for `merge`: scan left to right; `merge([5,5,5], (5,5), 9)` is `[9, 5]`.

In [4]:
def get_pair_counts(ids):
    """Counter of adjacent (a, b) pairs in ids."""
    raise NotImplementedError


def merge(ids, pair, new_id):
    """Replace each non-overlapping occurrence of `pair` (left to right) by new_id."""
    raise NotImplementedError


def learn_merges(text, n_merges):
    """Return the list of merged pairs, in the order learned. Merge k creates id 256 + k."""
    raise NotImplementedError


def bpe_encode(text, merges):
    """Bytes of text, then replay every merge in order."""
    raise NotImplementedError


def bpe_decode(ids, merges):
    """Map ids back to bytes (build id -> bytes from merges), join, decode UTF-8."""
    raise NotImplementedError

In [5]:
def t_pairs():
    assert get_pair_counts([1, 2, 1, 2, 3]) == Counter({(1, 2): 2, (2, 1): 1, (2, 3): 1}), "pair counts wrong"
    assert get_pair_counts([7]) == Counter(), "a single id has no pairs"

def t_merge():
    assert merge([1, 2, 1, 2, 3], (1, 2), 99) == [99, 99, 3]
    assert merge([5, 5, 5], (5, 5), 9) == [9, 5], "scan left to right, non-overlapping"
    assert merge([1, 2, 3], (9, 9), 50) == [1, 2, 3], "no match leaves ids unchanged"

train_text = raw_text[:20_000]
def t_first_merge():
    m = learn_merges("aaabdaaabac", 1)
    assert m == [(97, 97)], f"expected [(97, 97)] ('aa' is the most frequent pair), got {m}"

def t_roundtrip():
    merges = learn_merges(train_text, 100)
    assert len(merges) == 100
    for s in ["To be, or not to be", "Akwirw ier", "café 日本語 🦊", ""]:
        assert bpe_decode(bpe_encode(s, merges), merges) == s, f"round trip failed for {s!r}"

def t_compress():
    merges = learn_merges(train_text, 100)
    held_out = raw_text[20_000:25_000]
    n_bytes = len(held_out.encode("utf-8"))
    n_tok = len(bpe_encode(held_out, merges))
    assert n_tok < 0.85 * n_bytes, f"{n_tok} tokens for {n_bytes} bytes: merges should compress held-out text"

def t_deterministic():
    assert learn_merges(train_text, 30) == learn_merges(train_text, 30)

for label, fn in [("pair counts", t_pairs), ("merge", t_merge), ("first merge is 'aa'", t_first_merge),
                  ("round trip", t_roundtrip), ("compresses held-out text", t_compress),
                  ("deterministic", t_deterministic)]:
    check(label, fn)

⏳ pair counts: not implemented yet
⏳ merge: not implemented yet
⏳ first merge is 'aa': not implemented yet
⏳ round trip: not implemented yet
⏳ compresses held-out text: not implemented yet
⏳ deterministic: not implemented yet


In [6]:
# Once E1 passes: peek at what YOUR merges learned (skipped while E1 is unwritten)
try:
    my_merges = learn_merges(train_text, 60)
    vocab_bytes = {i: bytes([i]) for i in range(256)}
    for k, (a, b) in enumerate(my_merges):
        vocab_bytes[256 + k] = vocab_bytes[a] + vocab_bytes[b]
    print([vocab_bytes[256 + k] for k in range(0, 60, 4)])
except NotImplementedError:
    print("⏳ finish E1 first")

⏳ finish E1 first


## 3 · Next-token targets (your x / y shift)

The whole language-model objective: given tokens up to position *t*, predict token *t+1*.
Shift the sequence by one and every position becomes a training example **at once**.

In [7]:
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1 : context_size + 1]
print("x:", x)
print("y:", y)
for i in range(1, context_size + 1):
    print(repr(gpt2.decode(enc_sample[:i])), "---->", repr(gpt2.decode([enc_sample[i]])))

x: [12939, 13, 198, 198]
y: [13, 198, 198, 5962]
' resolved' ----> '.'
' resolved.' ----> '\n'
' resolved.\n' ----> '\n'
' resolved.\n\n' ----> 'First'


### ✍️ E2 · `make_xy(ids, context_size)`
Return `(x, y)` as lists where `y` is `x` shifted left by one. Raise `ValueError` if `ids` is too short to
produce a full `y` (you need `context_size + 1` ids).

In [8]:
def make_xy(ids, context_size):
    raise NotImplementedError

In [9]:
def t_xy():
    assert make_xy([10, 11, 12, 13, 14], 3) == ([10, 11, 12], [11, 12, 13])
    x, y = make_xy(enc_sample, 8)
    assert len(x) == len(y) == 8
    assert x[1:] == y[:-1], "y must be x shifted left by one"

def t_xy_short():
    try:
        make_xy([1, 2, 3], 3)
    except ValueError:
        return
    raise AssertionError("expected ValueError when len(ids) < context_size + 1")

check("x / y shift", t_xy)
check("too-short input raises ValueError", t_xy_short)

⏳ x / y shift: not implemented yet
⏳ too-short input raises ValueError: not implemented yet


False

## 4 · Sliding windows (your `GPTDatasetV1`)

Your class tokenizes everything, slides a window of `max_length` with step `stride`, and stores every window as a tensor.
`stride < max_length` gives overlapping windows (more examples, same text); `stride == max_length` gives none.

In [10]:
class GPTDatasetV1(Dataset):
    """Your dataset from notebook 01, unchanged."""

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i : i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1 : i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


ds = GPTDatasetV1(raw_text[:2000], gpt2, max_length=4, stride=1)
print(len(ds), "windows from 2000 characters")
for i in range(3):
    print(ds[i])

555 windows from 2000 characters
(tensor([ 5962, 22307,    25,   198]), tensor([22307,    25,   198,  8421]))
(tensor([22307,    25,   198,  8421]), tensor([  25,  198, 8421,  356]))
(tensor([  25,  198, 8421,  356]), tensor([ 198, 8421,  356, 5120]))


Note `ds[0]` and `ds[1]` overlap in 3 of 4 tokens. Count of windows is `ceil((n_tokens - max_length) / stride)`.

### ✍️ E3 · `sliding_windows(ids, max_length, stride)`
No tensors, no class: return a list of `(x, y)` list pairs with exactly the same windows as `GPTDatasetV1`.
Careful with where the range stops (the last window still needs one extra token for its `y`).

In [11]:
def sliding_windows(ids, max_length, stride):
    raise NotImplementedError

In [12]:
def t_windows_small():
    w = sliding_windows(list(range(10)), max_length=4, stride=2)
    assert w[0] == ([0, 1, 2, 3], [1, 2, 3, 4])
    assert w[1] == ([2, 3, 4, 5], [3, 4, 5, 6])
    assert all(len(x) == len(y) == 4 for x, y in w), "every window must be full length"
    assert w[-1][1][-1] <= 9, "ran past the end of ids"

def t_matches_notebook():
    txt = raw_text[:5000]
    for max_length, stride in [(4, 1), (8, 4), (16, 16)]:
        ref = GPTDatasetV1(txt, gpt2, max_length, stride)
        mine = sliding_windows(gpt2.encode(txt), max_length, stride)
        assert len(mine) == len(ref), f"window count {len(mine)} != {len(ref)} for ({max_length}, {stride})"
        for (x, y), rx, ry in zip(mine, ref.input_ids, ref.target_ids):
            assert x == rx.tolist() and y == ry.tolist()

check("small example", t_windows_small)
check("matches your GPTDatasetV1 exactly", t_matches_notebook)

⏳ small example: not implemented yet
⏳ matches your GPTDatasetV1 exactly: not implemented yet


False

## 5 · Memmap sampling (the upgrade over your dataset)

`GPTDatasetV1` stores every window as two int64 tensors. Memory = `n_windows x max_length x 8 bytes x 2`.
Look at how that scales:

In [13]:
n_tokens, T = len(enc_text), 256
for stride in (128, 1):
    n_win = (n_tokens - T) // stride
    gb = n_win * T * 8 * 2 / 1e9
    print(f"Shakespeare, T={T}, stride={stride:>3}: {n_win:>7,} windows -> {gb:7.2f} GB as tensors")
big = 40_000_000
print(f"TinyStories-scale ({big/1e6:.0f}M tokens), stride=1: {big * T * 16 / 1e9:,.0f} GB as tensors")
print(f"Same tokens as uint16 on disk:              {big * 2 / 1e6:,.0f} MB")

Shakespeare, T=256, stride=128:   2,638 windows ->    0.01 GB as tensors
Shakespeare, T=256, stride=  1: 337,769 windows ->    1.38 GB as tensors
TinyStories-scale (40M tokens), stride=1: 164 GB as tensors
Same tokens as uint16 on disk:              80 MB


🔮 **Predict:** is a stride-1 dataset of 40M tokens feasible in RAM? (No: ~164 GB for the tensors alone, see the output above.)

The fix: store all tokens **once** as `uint16` (2 bytes each, fine because vocab < 65536), memory-map the file, and pick
**random start positions** each step. Overlap comes for free from random starts, and there is no `stride`.

In [14]:
bin_path = Path(tempfile.mkdtemp()) / "train.bin"
np.array(enc_text, dtype=np.uint16).tofile(bin_path)
data = np.memmap(bin_path, dtype=np.uint16, mode="r")
print(bin_path, f"{bin_path.stat().st_size/1e6:.2f} MB;", "tokens:", len(data), "dtype:", data.dtype)
print(data[:10], "==", enc_text[:10])

/tmp/tmpvkr2nwok/train.bin 0.68 MB; tokens: 338025 dtype: uint16
[ 5962 22307    25   198  8421   356  5120   597  2252    11] == [5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11]


### ✍️ E4 · `sample_batch(data, batch_size, context_length, rng)`
Return `(x, y)` torch tensors of shape `(batch_size, context_length)` and dtype `torch.int64`
(embedding lookups need int64), with `y` = `x` shifted by one.

- pick `batch_size` random starts with `rng.integers(low, high, size=...)`. Think: what is the largest valid start so that `start + context_length + 1 <= len(data)`?
- slice the memmap for each start, `np.stack`, convert with `.astype(np.int64)` (memmap is uint16 and read-only), `torch.from_numpy`.

In [15]:
def sample_batch(data, batch_size, context_length, rng):
    # (B, T) inputs and (B, T) targets
    raise NotImplementedError

In [16]:
def t_shapes():
    x, y = sample_batch(data, 5, 16, np.random.default_rng(0))
    assert x.shape == y.shape == (5, 16), f"shape {tuple(x.shape)}"
    assert x.dtype == y.dtype == torch.int64, f"dtype {x.dtype}"

def t_shift():
    x, y = sample_batch(data, 8, 32, np.random.default_rng(1))
    assert torch.equal(x[:, 1:], y[:, :-1]), "y must be x shifted by one"

def t_contiguous():
    counting = np.arange(1000, dtype=np.uint16)  # token i is the number i, so windows are easy to verify
    x, y = sample_batch(counting, 6, 10, np.random.default_rng(2))
    assert torch.equal(y, x + 1) and torch.all(x[:, 1:] - x[:, :-1] == 1), "rows must be contiguous slices"

def t_bounds():
    counting = np.arange(20, dtype=np.uint16)
    x, y = sample_batch(counting, 500, 8, np.random.default_rng(3))
    assert y.max() <= 19, "window ran past the end of the data"
    assert x.min() == 0 and y.max() == 19, "with 500 draws every valid start should appear (check your high bound)"

def t_seeded():
    a = sample_batch(data, 4, 8, np.random.default_rng(7))
    b = sample_batch(data, 4, 8, np.random.default_rng(7))
    assert torch.equal(a[0], b[0]) and torch.equal(a[1], b[1]), "same seed must give the same batch"

for label, fn in [("shapes and dtype", t_shapes), ("y is x shifted", t_shift),
                  ("rows are contiguous slices", t_contiguous), ("never runs off the end", t_bounds),
                  ("seeded determinism", t_seeded)]:
    check(label, fn)

⏳ shapes and dtype: not implemented yet
⏳ y is x shifted: not implemented yet
⏳ rows are contiguous slices: not implemented yet
⏳ never runs off the end: not implemented yet
⏳ seeded determinism: not implemented yet


In [17]:
# Once E4 passes: a real batch, decoded back to text
try:
    xb, yb = sample_batch(data, 2, 12, np.random.default_rng(0))
    for row in range(2):
        print(repr(gpt2.decode(xb[row].tolist())), "\n   ->", repr(gpt2.decode(yb[row].tolist())))
except NotImplementedError:
    print("⏳ finish E4 first")

⏳ finish E4 first


## 6 · Embeddings (your last two sections)

Token ids are just labels. The model needs vectors it can learn. `nn.Embedding(V, C)` is a `(V, C)` table; looking up id `i`
returns row `i`. Your notebook used `vocab_size=6, output_dim=3`:

In [18]:
torch.manual_seed(123)
emb = torch.nn.Embedding(6, 3)
print(emb.weight)
input_ids = torch.tensor([2, 3, 5, 1])
print(emb(input_ids))                      # shape (4, 3): rows 2, 3, 5, 1 of the table
print(torch.equal(emb(input_ids), emb.weight[input_ids]))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
True


It is *only* indexing. The equivalent view: a one-hot vector times the table (a matmul that selects one row), which is
why embeddings are trainable like any linear layer, just without wasting compute on the zeros.

### ✍️ E5 · Two ways to embed
Given a weight matrix `W` of shape `(V, C)` and an integer tensor `ids` of **any** shape `(...)`:
- `lookup(W, ids)` -> shape `(..., C)` by indexing
- `lookup_onehot(W, ids)` -> same result via `torch.nn.functional.one_hot(ids, V)` and a matmul (watch the dtype)

In [19]:
def lookup(W, ids):
    raise NotImplementedError


def lookup_onehot(W, ids):
    raise NotImplementedError

In [20]:
def t_lookup():
    W = torch.randn(6, 3)
    ids = torch.tensor([2, 3, 5, 1])
    assert lookup(W, ids).shape == (4, 3)
    assert torch.equal(lookup(W, ids), torch.nn.Embedding.from_pretrained(W)(ids)), "differs from nn.Embedding"

def t_onehot():
    W = torch.randn(50, 8)
    ids = torch.randint(0, 50, (3, 7))
    out = lookup_onehot(W, ids)
    assert out.shape == (3, 7, 8), f"shape {tuple(out.shape)}"
    assert torch.allclose(out, lookup(W, ids)), "one-hot matmul must equal indexing"

check("indexing lookup", t_lookup)
check("one-hot matmul equals lookup", t_onehot)

⏳ indexing lookup: not implemented yet
⏳ one-hot matmul equals lookup: not implemented yet


False

### Positions
Attention has no sense of order, so we add a second table indexed by **position** (0..T-1), also learned (GPT-2 style).
Your notebook did this at GPT-2 scale: `(B, T) -> (B, T, 256)`.

🔮 **Predict:** shapes of `tok_emb`, `pos_emb`, and their sum for `B=8, T=4, C=256`. Why does `+` work between different shapes?

### ✍️ E6 · `input_embeddings(tok_W, pos_W, ids)`
`tok_W` is `(V, C)`, `pos_W` is `(context_length, C)`, `ids` is `(B, T)` with `T <= context_length`.
Return `(B, T, C)` = token vector + position vector. Use broadcasting; only the first `T` rows of `pos_W` are needed.

In [21]:
def input_embeddings(tok_W, pos_W, ids):
    # ids (B, T) -> (B, T, C)
    raise NotImplementedError

In [22]:
def t_shape():
    tok_W, pos_W = torch.randn(100, 16), torch.randn(32, 16)
    ids = torch.randint(0, 100, (8, 4))
    assert input_embeddings(tok_W, pos_W, ids).shape == (8, 4, 16)

def t_loop_oracle():
    tok_W, pos_W = torch.randn(100, 16), torch.randn(32, 16)
    ids = torch.randint(0, 100, (3, 5))
    out = input_embeddings(tok_W, pos_W, ids)
    for b in range(3):
        for t in range(5):
            assert torch.allclose(out[b, t], tok_W[ids[b, t]] + pos_W[t]), f"mismatch at b={b}, t={t}"

def t_position_matters():
    tok_W, pos_W = torch.randn(10, 4), torch.randn(8, 4)
    out = input_embeddings(tok_W, pos_W, torch.tensor([[3, 3, 3]]))
    assert not torch.allclose(out[0, 0], out[0, 1]), "same token at different positions must differ"

for label, fn in [("shape (B, T, C)", t_shape), ("matches explicit loop", t_loop_oracle),
                  ("position changes the vector", t_position_matters)]:
    check(label, fn)

⏳ shape (B, T, C): not implemented yet
⏳ matches explicit loop: not implemented yet
⏳ position changes the vector: not implemented yet


In [23]:
# Parameter cost of the input stage, ours vs GPT-2's vocab, at d_model = 256
C = 256
for name, V, T_ctx in [("our small run", 4096, 256), ("GPT-2 vocab", 50257, 256)]:
    print(f"{name:>14}: token table {V * C / 1e6:5.2f}M + position table {T_ctx * C / 1e3:.0f}K")

 our small run: token table  1.05M + position table 66K
   GPT-2 vocab: token table 12.87M + position table 66K


## Wrap-up

You now have, by your own hand: BPE training (E1), the LM target shift (E2), the sliding window (E3), the memmap
batch sampler (E4), embedding lookup (E5) and token+position embeddings (E6).

**Interview drill:** *Why random-start windows instead of strided ones? Why is `ln(V)` the loss of an untrained model?
Why does a bigger vocab cost parameters but save sequence length?*

**Next in the project:** Phase 2 turns E4 into `data/dataset.py` (plus the `prepare-data` pipeline), Phase 3 uses E5/E6 as the first
lines of `GPT.forward`, and the tests you just ran become the oracle for your later rewrites in `docs/LEARNING_PLAN.md`.